In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# 92 — Self-Reflecting Research Agent

## Goal
Build an agent that **searches** for information, **drafts** a
report, **critiques** its own draft, then **revises** it —
all automatically in a loop.

## What You'll Learn
| Concept | Detail |
|---------|--------|
| **Self-reflection** | Agent evaluates its own output quality |
| **Iterative refinement** | Draft → Critique → Revise loop |
| **Conditional looping** | LangGraph edges that loop back |
| **Quality gating** | Only exit the loop when quality is sufficient |

## Stack
- `langgraph` — stateful graph with critique-revise loop
- `langchain-ollama` — local LLM
- Ollama `qwen3.5:9b`

In [ ]:
import os, warnings, json, re
os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from typing import TypedDict
from langgraph.graph import StateGraph, END
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage

llm = ChatOllama(model="qwen3.5:9b", temperature=0)
print("All imports OK")

## Step 1 — Simulated Knowledge Base

In [ ]:
# Simulated search results the agent can query
KNOWLEDGE_BASE = {
    "transformer architecture": [
        "Transformers were introduced in 'Attention Is All You Need' (Vaswani et al., 2017).",
        "Self-attention allows the model to weigh different parts of the input sequence.",
        "Transformers use positional encoding since they lack recurrence.",
        "Multi-head attention runs several attention functions in parallel."
    ],
    "scaling laws": [
        "Kaplan et al. (2020) showed model performance scales as a power law with parameters, data, and compute.",
        "Chinchilla (Hoffmann et al., 2022) showed many LLMs were undertrained on data.",
        "Scaling laws suggest a 10x larger model needs ~3x improvement in loss.",
        "Optimal training tokens scale linearly with model parameters (Chinchilla optimal)."
    ],
    "fine-tuning techniques": [
        "LoRA (Hu et al., 2021) adds low-rank adapters to frozen weights, reducing trainable params by 99%.",
        "QLoRA combines 4-bit quantization with LoRA for memory-efficient fine-tuning.",
        "Full fine-tuning updates all parameters but requires significant GPU memory.",
        "RLHF uses human preferences to align model outputs via reward modeling."
    ],
    "inference optimization": [
        "KV-cache stores past key-value pairs to avoid recomputation during autoregressive generation.",
        "Speculative decoding uses a small draft model to propose tokens verified by the large model.",
        "Quantization (GPTQ, AWQ) reduces model weights to 4-bit with minimal quality loss.",
        "vLLM uses PagedAttention for efficient KV-cache memory management."
    ]
}

def search_knowledge(query: str) -> list[str]:
    """Simple keyword search over the knowledge base."""
    results = []
    query_lower = query.lower()
    for topic, facts in KNOWLEDGE_BASE.items():
        if any(word in query_lower for word in topic.split()):
            results.extend(facts)
    if not results:
        # Fallback: return everything
        for facts in KNOWLEDGE_BASE.values():
            results.extend(facts)
    return results

print(f"Knowledge base: {len(KNOWLEDGE_BASE)} topics, {sum(len(v) for v in KNOWLEDGE_BASE.values())} facts")

## Step 2 — Define the Agent Graph

```
search → draft → critique → [good enough?]
                               → YES: finalize → END
                               → NO:  revise → critique (loop)
```

In [ ]:
class ResearchState(TypedDict):
    topic: str
    search_results: list
    draft: str
    critique: str
    score: int
    revision_count: int
    final_report: str

MAX_REVISIONS = 2  # Cap revisions to avoid infinite loops
QUALITY_THRESHOLD = 7  # Score out of 10 to pass

print(f"Max revisions: {MAX_REVISIONS}, Quality threshold: {QUALITY_THRESHOLD}/10")

In [ ]:
def search_node(state: ResearchState) -> ResearchState:
    """Search the knowledge base for relevant information."""
    print(f"\n🔍 Searching for: {state['topic']}")
    results = search_knowledge(state["topic"])
    print(f"   Found {len(results)} facts")
    return {**state, "search_results": results}

def draft_node(state: ResearchState) -> ResearchState:
    """Draft a research report from search results."""
    print("\n📝 Drafting report...")
    facts = "\n".join(f"- {f}" for f in state["search_results"])
    
    msg = llm.invoke([
        SystemMessage(content="""Write a concise research report (250-350 words) based on
the provided facts. Include:
1. An introduction stating the topic
2. Key findings organized logically
3. A brief conclusion
Cite facts where appropriate. Be educational and clear.
Do NOT use thinking tags."""),
        HumanMessage(content=f"Topic: {state['topic']}\n\nFacts:\n{facts}")
    ])
    
    draft = msg.content
    if "<think>" in draft:
        draft = draft.split("</think>")[-1].strip()
    
    print(f"   Draft length: {len(draft)} chars")
    return {**state, "draft": draft}

print("Nodes: search, draft defined")

In [ ]:
def critique_node(state: ResearchState) -> ResearchState:
    """Self-critique the current draft."""
    print(f"\n🔎 Self-critiquing (revision {state['revision_count']})...")
    
    msg = llm.invoke([
        SystemMessage(content="""You are a strict academic reviewer. Evaluate this research report.

Score it 1-10 on overall quality considering:
- Accuracy and completeness
- Logical flow and organization
- Clarity of writing
- Use of evidence/citations

Return EXACTLY this JSON format:
{"score": <1-10>, "strengths": ["..."], "weaknesses": ["..."], "suggestions": ["..."]}

Return ONLY the JSON, no other text."""),
        HumanMessage(content=state["draft"])
    ])
    
    raw = msg.content
    if "<think>" in raw:
        raw = raw.split("</think>")[-1].strip()
    if "```" in raw:
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
        raw = raw.strip()
    
    try:
        start = raw.find("{")
        end = raw.rfind("}") + 1
        critique_data = json.loads(raw[start:end])
        score = int(critique_data.get("score", 5))
        critique_text = json.dumps(critique_data, indent=2)
    except (json.JSONDecodeError, ValueError):
        score = 5
        critique_text = raw[:500]
    
    print(f"   Score: {score}/10")
    return {**state, "critique": critique_text, "score": score}

print("Node: critique defined")

In [ ]:
def revise_node(state: ResearchState) -> ResearchState:
    """Revise the draft based on critique feedback."""
    print(f"\n✏️ Revising draft (attempt {state['revision_count'] + 1})...")
    
    msg = llm.invoke([
        SystemMessage(content="""Revise this research report based on the critique below.
Address ALL weaknesses and incorporate the suggestions.
Keep the report 250-350 words. Improve quality significantly.
Do NOT use thinking tags."""),
        HumanMessage(content=f"CURRENT DRAFT:\n{state['draft']}\n\nCRITIQUE:\n{state['critique']}")
    ])
    
    revised = msg.content
    if "<think>" in revised:
        revised = revised.split("</think>")[-1].strip()
    
    return {**state, "draft": revised, "revision_count": state["revision_count"] + 1}

def finalize_node(state: ResearchState) -> ResearchState:
    """Accept the draft as final."""
    print(f"\n✅ Report finalized (score: {state['score']}/10, revisions: {state['revision_count']})")
    return {**state, "final_report": state["draft"]}

# Router: quality gate
def quality_gate(state: ResearchState) -> str:
    if state["score"] >= QUALITY_THRESHOLD:
        return "finalize"
    if state["revision_count"] >= MAX_REVISIONS:
        print(f"   ⚠️ Max revisions reached, accepting current draft")
        return "finalize"
    return "revise"

print("Nodes: revise, finalize, quality_gate defined")

In [ ]:
# Build the graph
graph = StateGraph(ResearchState)

graph.add_node("search", search_node)
graph.add_node("draft", draft_node)
graph.add_node("critique", critique_node)
graph.add_node("revise", revise_node)
graph.add_node("finalize", finalize_node)

graph.set_entry_point("search")
graph.add_edge("search", "draft")
graph.add_edge("draft", "critique")
graph.add_conditional_edges("critique", quality_gate, {
    "finalize": "finalize",
    "revise": "revise"
})
graph.add_edge("revise", "critique")  # Loop back!
graph.add_edge("finalize", END)

research_app = graph.compile()
print("Self-reflecting research graph compiled!")

In [ ]:
# Run the agent
result = research_app.invoke({
    "topic": "Modern LLM training: transformers, scaling laws, and fine-tuning",
    "search_results": [],
    "draft": "",
    "critique": "",
    "score": 0,
    "revision_count": 0,
    "final_report": ""
})

print("\n" + "=" * 60)
print("FINAL RESEARCH REPORT")
print(f"Score: {result['score']}/10 | Revisions: {result['revision_count']}")
print("=" * 60)
print(result["final_report"])

In [ ]:
# Show the last critique
print("LAST CRITIQUE:")
print(result["critique"])

## 🧠 Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| **Self-reflection** | LLM critiques its own draft with scoring |
| **Quality gate** | `add_conditional_edges` routes to revise or finalize |
| **Looping graph** | `revise → critique` edge creates a refinement loop |
| **Max iterations** | `MAX_REVISIONS` prevents infinite loops |
| **Threshold** | `QUALITY_THRESHOLD` defines "good enough" |

## Self-Reflection Loop
```
Search → Draft → Critique ─┐
                ↑    │      │
                │    │  score >= 7?
                │    │      │
                │    NO     YES
                │    ↓      ↓
                └─ Revise  Finalize → END
```

## Extending This Project
- Real web search via API (Tavily, DuckDuckGo)
- Multi-reviewer panel (different critique perspectives)
- Store revision history for analysis
- Human-in-the-loop critique at specific iterations